# Crossy Road - przestrzenie ciągłe

Marcin Wolder i Stanisław Wojtas

**Cel projektu**: Analiza procesu uczenia przez wzmacnianie w ciągłej przestrzeni stanów na przykładzie autorskiego środowiska CrossyRoadEnv z wykorzystaniem algorytmu PPO (Proximal Policy Optimization) z biblioteki stable-baselines3

### Charakterystyka środowiska i algorytmu


CrossyRoadEnv to środowisko z dyskretną przestrzenią akcji Discrete(5) (ruch w 4 kierunkach oraz akcja oczekiwania) i ciągłą przestrzenią stanów Box(float32). Funkcja nagrody promuje zbliżanie się do linii mety (nagroda za postęp, nagroda za ukończenie planszy) i surowo karze zachowania niebezpieczne (kolizje z pojazdami, ruch wstecz, ignorowanie ryzyka).

PPO to zaawansowany algorytm optymalizacji polityki należący do metod gradientowych (Policy Gradient). Wykorzystuje technikę przycinania wskaźnika prawdopodobieństwa zmian (clipping objective), co zapobiega zbyt gwałtownym i destrukcyjnym zmianom parametrów sieci w jednej iteracji, zapewniając stabilność numeryczną podczas optymalizacji

### Opis wejścia i wyjścia sieci

**Wejście** to ciągły wektor o wymiarowości 4 + (liczba pasów * 3) + (2 * obs_radius + 1)^2. Składa się z: pozycji znormalizowanej gracza (X, Y), binarnych flag przewidywanego ruchu i kolizji krok w przód, relatywnej odległości, prędkości i kierunku najbliższego pojazdu na każdym z pasów oraz lokalnej siatki zajętości ( occupancy map) o promieniu 2 wokół gracza.

**Wyjście**: Dyskretny rozkład prawdopodobieństwa dla 5 możliwych decyzji agenta: 0: up, 1: down, 2: left, 3: right, 4: wait.

### Schematy architektury sieci (PPO, MlpPolicy)

Wszystkie warianty maja wspolny ekstraktor cech (wejscie 59), a potem dwie niezalezne galezie: policy i value. W warstwach ukrytych aktywacja `tanh`.

**Konfiguracja 1: baseline_64x64**

```mermaid
flowchart LR
    A[Wejscie: 59] --> B1[Policy Dense 64, tanh]
    B1 --> B2[Policy Dense 64, tanh]
    B2 --> B3[Policy logity: 5]
    A --> C1[Value Dense 64, tanh]
    C1 --> C2[Value Dense 64, tanh]
    C2 --> C3[Value: 1]
```

**Konfiguracja 2: deep_128x128**

```mermaid
flowchart LR
    A[Wejscie: 59] --> B1[Policy Dense 128, tanh]
    B1 --> B2[Policy Dense 128, tanh]
    B2 --> B3[Policy logity: 5]
    A --> C1[Value Dense 128, tanh]
    C1 --> C2[Value Dense 128, tanh]
    C2 --> C3[Value: 1]
```

**Konfiguracja 3: fast_64x64**

```mermaid
flowchart LR
    A[Wejscie: 59] --> B1[Policy Dense 64, tanh]
    B1 --> B2[Policy Dense 64, tanh]
    B2 --> B3[Policy logity: 5]
    A --> C1[Value Dense 64, tanh]
    C1 --> C2[Value Dense 64, tanh]
    C2 --> C3[Value: 1]
```

### Uruchomienie treningu i Eksperymentów

Trening DQN:

```bash
uv run train.py --timesteps 100000 --seed 42 --max-steps 500
```

Eksperymenty PPO (3 konfiguracje x 10 seedów):

```bash
uv run run_experiments.py
```

Wyniki znajdziesz w katalogu `artifacts/` (modele, `monitor.csv`, `eval_summary.json`, `learning_curves.png`).

### Zestawienie hiperparametrow

Czytelna tabela podsumowujaca parametry konfiguracyjne przekazane do algorytmu PPO:

| Konfiguracja | learning_rate | batch_size | n_steps | gamma | net_arch |
| --- | ---: | ---: | ---: | ---: | --- |
| baseline_64x64 | 3e-4 | 64 | 2048 | 0.99 | [64, 64] |
| deep_128x128 | 1e-4 | 128 | 1024 | 0.98 | [128, 128] |
| fast_64x64 | 5e-4 | 256 | 512 | 0.995 | [64, 64] |

### Zbiorcze krzywe uczenia

![Zbiorcze krzywe uczenia](artifacts/experiments/learning_curves.png)

### Analiza Wynikow i Pomiaru Czasu

**Czas**: Sredni czas jednego kroku srodowiska: 0.239 ms. Estymowany czas pelnego epizodu (500 krokow): 119.48 ms.

**Uzasadnienie**: Krzywe uczenia dla deep_128x128 i fast_64x64 rosna szybciej niz baseline_64x64, co wskazuje na bardziej efektywne uczenie w pierwszej polowie treningu. Najwyzsza nagrode koncowa osiaga deep_128x128, ktore utrzymuje najlepszy poziom sredniej nagrody w koncowce. Najmniejsze odchylenie standardowe widoczne jest dla fast_64x64, co sugeruje bardziej stabilny przebieg uczenia. Jednak różnice pomiędzy deep_128x128 a fast_64x64 są niewielkie i można założyć, że sieci uzyskały podobne wyniki. Jedynie baseline wskazuje znacząco gorsze wyniki.

### Analiza najlepszego agenta w trybie deterministycznym

In [1]:
from pathlib import Path
import json

summary_path = Path("artifacts/experiments/config_3/seed_4/eval_summary.json")
with summary_path.open("r", encoding="utf-8") as handle:
    summary = json.load(handle)

summary

{'config_index': 3,
 'config_name': 'fast_64x64',
 'seed': 4,
 'episodes': 10,
 'mean_deterministic_reward': -18.904999999999784}

### Porownanie agenta z krzywa uczenia (wnioski koncowe)


Wartosc `mean_deterministic_reward` z testu bez eksploracji wynosi ok. -19.74 i jest zblizona do koncowych wartosci widocznych na krzywych uczenia, gdzie wciaz wystepowal element stochastyczny. Oznacza to, ze po wylaczeniu losowosci agent utrzymuje podobny poziom jakosci, ale z mniejszym rozrzutem zachowania.

Zablokowanie eksploracji (wybor zawsze najlepszej, deterministycznej akcji sieci przez `deterministic=True`) zazwyczaj skutkuje podniesieniem sredniej nagrody i eliminacja losowych, blednych krokow (np. niepotrzebnego wejscia pod kola samochodu), co potwierdza, ze siec skutecznie zakodowala strategiczna mape decyzji, a nie dziala chaotycznie.